### Timing: Depending on target number and system RAM/CPU, 30 min - 12 h or more

environment: openFISH_probe  

This notebook is consist of one main section for probe design and two optional helper section for input preparation.

Five input are needed for the probe design:  
1. Input file (see below for the right format);
2. blastn refseq_rna database
3. Using padlock (see part2 about how to generate one)
4. bridge sequences
5. (Optional) Black ground gene list

**mRNA, cDNA, genome, artificial sequences (like GFP/HA sequences that are not in the genome) are support for probe design in this notebook**

mRNA/cDNA format: Gad2

Artificial sequences format：Artificial_XXX

**Please prepare a input file format like below and saved as CSV format:**  
**CRITICAL:** The dataframe is just an format example, for real situation, **DO NOT** include multiple kinds of target in one CSV file. One CSV input should only has one kind of target.  
**Note:** If you leave seq column empty, remember to fill in the "Entrez_api_key" and "Entrez_email" parameters and keep internet connected.

| bridge_id | target | seq |
| ----------- | ----------- | ----------- |
| PadLock_001 | Gad2 |   |
| PadLock_001 | Artificial_GFP | ACGATGCTAGCTAGCATGCTACGTCGATCGT |

<div class="alert alert-block alert-warning">
<b>Maker sure "taxdb.btd", "taxdb.bti" and "taxonomy4blast.sqlite3" are under the same folder with this notebook</b>
</div>

# Main section: Probe generator

## Change input file name

In [1]:
input_filename = "Example.csv"

## Run the following cells to design gene probes

In [6]:
import numpy as np
import pandas as pd
import os
from openDesign.generator import Para, gene_generator
from openDesign.generator_multi import multigene_generator
from openDesign.utils import generate_order_form

pathsplit = os.path.split(input_filename)
rootpath = pathsplit[0]
file_output = os.path.join(rootpath, input_filename.split(".")[0] + "_results.csv")
file_output_order = os.path.join(rootpath, input_filename.split(".")[0] + "_results_order.csv")

def input_format(x):
    return x.strip().replace('\n', '')
genes = pd.read_csv(input_filename, sep=",")
genes.columns = ["bridge_id", "target", "seq"]
genes.loc[:,"bridge_id"] = genes.loc[:,"bridge_id"].apply(input_format)
try:
    genes.loc[:,"seq"] = genes.loc[:,"seq"].apply(input_format)
except:
    pass
genes.loc[:,"target"] = genes.loc[:,"target"].apply(input_format)
print(str(genes.shape))
genes

(1, 3)


,bridge_id,target,seq
0,PadLock_001,Gad2,NaN


In [3]:
para = Para(
        gc_thred=[20,80],  # GC range, in %
        sCelsius = 47,  # Hybridization temperature
        nCelsius = 37, # Allowed highest Tm for unspecific binding
        df_conc = 30, # formamide conc. in %
        na_conc=0.390,  # monovalent ion conc. in M
        mg_conc=0.0, # divalent ion conc. in M
        bseqs_path = "./SEQUENCES/Bridge_sequences.csv", # Bridge sequences file path
        UsingPadfile = "./SEQUENCES/UsingPad.fa", # Using padlock file path, used for inter-padlock alignment. Trim or Expand this file if you're using less or more padlocks, correspondingly
        TaxonomyID = "10090", # Species taxonomy ID, has to correspondant to species. Check in NCBI
        species = 'Mus musculus', # Species name, has to be correspondant to taxonomy ID. Check in NCBI
        P1evalue = 100, # Highest evalue considered when BLASTn P1, lower this value if you do not have enough outputs. 50,10,5
        P2evalue =5, # Highest evalue considered when BLASTn P2
        strand="minus", # The strand of designed gene probes. If target mRNA, fill in minus, if target cDNA, fill in plus. 
        BLACK_FA = "./SEQUENCES/BLACK_LIST_FULL.fa", # BLACK LIST, sequences that have been tested unreliable for unknown reason. tRNA sequences are also included.
        gtype = 'gene', # Auto download the region of gene, as input for NCBI. can chooase from: gene,CDS,exon
        max_threads = 64, 
        max_memory = 256,
        # Some unspecific binding for some genes in one tissue may not be harmful, you can input a gene blacklist to check that whether your designed probes have any unspecific binds on blacklist genes
        # background_list = "./SEQUENCES/Mouse_Brain_Background.csv",
        # background_list = ['Gad2', 'Gad1', ..., 'Gpadh']
        background_list = None,
        
        # NCBI's api key and email，used for sequences auto download
        Entrez_api_key = "",
        Entrez_email = "",
        # executable blastn binary path
        blastn_path = '/path/to/blastn',
        # refseq_rna path
        ref_seq_path = "/path/to/refseq_rna"
        )

results = multigene_generator(genes, para)
results.to_csv(file_output, sep=",")

  0%|                                                                                                                                                                                                                             | 0/1 [00:00<?, ?it/s]

[INFO]Generating probes for Gad2
[INFO]Downloading cDNA for Gad2
[INFO]8569 candidate probes
[INFO]2998 probes passed basic filter
[INFO]816 probes passed p1 blastn filter
[INFO]401 probes passed p2 blastn filter
[INFO]398 P1 probes passed black list filter
[INFO]398 P2 probes passed black list filter
[INFO]396 probes passed P1 system-reaction filter
[INFO]376 probes passed P2 system-reaction filter
[INFO]Preparing Indicator for Efficiency Calculatioin...
[INFO]Calculating Binding Efficiency...


[INFO]376 pairs of probes are selected to make complete sequence


---

---

## Pick probes manually

After above generation, a file will be output into the current directory called "[input_file_name]_results.csv".  
Open it and choose the indices (the column on the left if the 'name' column) for each gene.  
After selection, run the cell and a '[input_file_name]_results_order.csv' will be saved in the current directory.

In [4]:
KEEP_INDICES = [
    0,3
    ] # input index of selected probes manually
results = results.loc[KEEP_INDICES, :].copy()
results = results.reset_index(drop = True)

final_df = generate_order_form(results)
final_df.to_csv(file_output_order, sep=",", index = False)
final_df

___

___

### Helper section: BLACK LIST Generation

In [1]:
import pandas as pd

In [2]:
df = pd.read_table("BLACK_LIST.tsv", header=None) # This is a file consists of bad gene probes with two columns without title. First column is Name, second column is sequence. Tab seperated.
df.columns = ['Name', 'Seq']

In [3]:
import re

def contains_pattern_A(s):
    pattern = r'_\d{1,2}A_'
    return re.search(pattern, s) is not None

def contains_pattern_B(s):
    pattern = r'_\d{1,2}B_'
    return re.search(pattern, s) is not None

In [4]:
with open("SEQUENCES/BLACK_LIST.fa", "w") as handle:
    for i in df.index:
        handle.write(f">{df.loc[i, 'Name']}\n")
        if contains_pattern_A(df.loc[i, 'Name']):
            handle.write(f"{df.loc[i, 'Seq'][-30:]}\n")
        elif contains_pattern_B(df.loc[i, 'Name']):
            handle.write(f"{df.loc[i, 'Seq'][0:30]}\n")

In [5]:
cat SEQUENCES/BLACK_LIST.fa SEQUENCES/mm39_tRNA.fa > SEQUENCES/BLACK_LIST_FULL.fa

### Helper section: BLACK gene list file Generation

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

In [ ]:
adata = sc.read_h5ad("/path/to/reference/scRNAseq.h5ad")
adata

In [4]:
df = adata.to_df()

In [5]:
def get_median_exp(column):
    return column[column > 0].median()

In [6]:
result = df.apply(lambda col: get_median_exp(col), axis = 0)

In [7]:
result = result.dropna()

In [8]:
result = result.sort_values(ascending=False)

In [9]:
background_list = result[result > np.percentile(result, 10)].index.to_list()

In [10]:
pd.DataFrame(background_list).to_csv("SEQUENCES/gene_Background.csv")